In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
import torch, rdkit
import os
from pathlib import Path
import sys
def _is_models_root(p: Path) -> bool:
    return (p / "utils").is_dir() and (p / "models").is_dir() and (p / "notebooks").is_dir() and (p / "data").is_dir()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if _is_models_root(p):
            return p
        cand = p / "MY_PAPER_RELATED" / "MODELS"
        if _is_models_root(cand):
            return cand
    raise FileNotFoundError("Could not locate MODELS project root")
PROJECT_ROOT = resolve_project_root()
os.environ["MODELS_VARIANT"] = "Encoder_Only"
os.environ["MODELS_FULL_DATA_TRAINING"] = "0"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rdkit import Chem
from utils.utils import *
from utils.dataloader import *
from utils.loss_fn import *
from utils.eval import *

from tqdm import trange
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test = sfp.encoder_psmiles("CC(CCCNC(=O)C(C)OC(=O)[*])O[*]")
print(test)
print(list(sfp.split_selfies(test)))
vocab = dataset.vocab
index_to_token = {idx: token for token, idx in vocab.items()}
print("Split summary:")
print(split_summary.to_string(index=False))
print("Split leakage counts:", split_info["leak_counts"])
print(f"Split tag: {SPLIT_TAG}")
print(f"Full-data training mode: {FULL_DATA_TRAINING}")
print(f"Dataset sizes - train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")


In [ ]:
from models.Trans import Encoder_Only

mode = "Encoder_Only"
model_cls = Encoder_Only
model = model_cls().to(device)

PAD_IDX = dataset.vocab['[PAD]']
loss_fn = nn.CrossEntropyLoss(reduction="mean", ignore_index=PAD_IDX)


In [ ]:
lr = 3e-4
from torch.optim import AdamW


def set_data_context(ctx):
    global dataset, train_dataset, val_dataset, test_dataset, trainval_dataset
    global train_dataloader, val_dataloader, test_dataloader, trainval_dataloader, eval_dataloader
    global train_indices, val_indices, test_indices, trainval_indices
    global fold_assignment, split_summary, split_info, SPLIT_TAG, vocab, index_to_token, PAD_IDX

    dataset = ctx["dataset"]
    train_dataset = ctx["train_dataset"]
    val_dataset = ctx["val_dataset"]
    test_dataset = ctx["test_dataset"]
    trainval_dataset = ctx["trainval_dataset"]
    train_dataloader = ctx["train_dataloader"]
    val_dataloader = ctx["val_dataloader"]
    test_dataloader = ctx["test_dataloader"]
    trainval_dataloader = ctx["trainval_dataloader"]
    eval_dataloader = ctx["eval_dataloader"]
    train_indices = ctx["train_indices"]
    val_indices = ctx["val_indices"]
    test_indices = ctx["test_indices"]
    trainval_indices = ctx["trainval_indices"]
    fold_assignment = ctx["fold_assignment"]
    split_summary = ctx["split_summary"]
    split_info = ctx["split_info"]
    SPLIT_TAG = ctx["split_tag"]
    vocab = dataset.vocab
    index_to_token = {idx: token for token, idx in vocab.items()}
    PAD_IDX = dataset.vocab["[PAD]"]
    return ctx


def init_model_bundle():
    model = model_cls().to(device)
    loss_fn = nn.CrossEntropyLoss(reduction="mean", ignore_index=dataset.vocab["[PAD]"])
    optim = AdamW(model.parameters(), lr=lr)
    return model, loss_fn, optim


model, loss_fn, optim = init_model_bundle()


In [ ]:
import copy
import datetime
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.error')

MAX_EPOCHS = 2000
VAL_EVERY = 10
EARLY_STOP_PATIENCE = 20
EARLY_STOP_MIN_DELTA = 1e-4
RUN_FULL_CV = True
RUN_FINAL_TRAIN_ON_TRAINVAL = True

VARIANT_NAME = os.environ.get("MODELS_VARIANT", mode)
CV_VAL_FOLDS = [f for f in range(SPLIT_N_FOLDS) if f != TEST_FOLD]
CV_CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "cv_es" / VARIANT_NAME
CV_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def train_encoder_one_epoch(model, loss_fn, optim, loader):
    model.train()
    batchloss = 0.0
    token_total = 0
    token_correct = 0
    for batch in loader:
        sm_enc, sm_dec_in, sm_dec_out, cond_scalar = [t.to(device) for t in batch]
        optim.zero_grad()
        attn_mask = sm_dec_in == PAD_IDX
        output = model(sm_dec_in, cond_scalar, attn_mask=attn_mask)
        loss = loss_fn(output.reshape(-1, dataset.vocab_size), sm_dec_out.view(-1))
        loss.backward()
        optim.step()
        batchloss += float(loss.item())
        with torch.no_grad():
            pred_tok = output.argmax(dim=-1)
            nonpad_mask = sm_dec_out != PAD_IDX
            token_total += int(nonpad_mask.sum().item())
            token_correct += int(((pred_tok == sm_dec_out) & nonpad_mask).sum().item())
    return {
        "loss": batchloss / max(1, len(loader)),
        "tok_acc": token_correct / max(1, token_total),
    }


def evaluate_encoder_loss(model, loss_fn, loader):
    if loader is None or len(loader) == 0:
        return {"loss": float("nan"), "tok_acc": float("nan")}
    model.eval()
    batchloss = 0.0
    token_total = 0
    token_correct = 0
    with torch.no_grad():
        for batch in loader:
            sm_enc, sm_dec_in, sm_dec_out, cond_scalar = [t.to(device) for t in batch]
            attn_mask = sm_dec_in == PAD_IDX
            output = model(sm_dec_in, cond_scalar, attn_mask=attn_mask)
            loss = loss_fn(output.reshape(-1, dataset.vocab_size), sm_dec_out.view(-1))
            batchloss += float(loss.item())
            pred_tok = output.argmax(dim=-1)
            nonpad_mask = sm_dec_out != PAD_IDX
            token_total += int(nonpad_mask.sum().item())
            token_correct += int(((pred_tok == sm_dec_out) & nonpad_mask).sum().item())
    return {
        "loss": batchloss / max(1, len(loader)),
        "tok_acc": token_correct / max(1, token_total),
    }


def save_encoder_checkpoint(path, model, epoch, val_metrics, split_tag):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "epoch": int(epoch),
            "val_metrics": val_metrics,
            "split_tag": split_tag,
            "variant": VARIANT_NAME,
            "mode": mode,
        },
        path,
    )


def load_encoder_checkpoint(path, model):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    return ckpt


def train_encoder_split(run_name, ctx, max_epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE, val_every=VAL_EVERY, min_delta=EARLY_STOP_MIN_DELTA):
    set_data_context(ctx)
    local_model, local_loss_fn, local_optim = init_model_bundle()
    best_val = float("inf")
    best_epoch = -1
    stale_checks = 0
    history = []
    ckpt_path = CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_{run_name}_cv_es_best.pth"
    progress = tqdm(range(max_epochs), desc=run_name)

    for epoch_idx in progress:
        train_metrics = train_encoder_one_epoch(local_model, local_loss_fn, local_optim, train_dataloader)
        should_validate = len(val_dataloader) > 0 and ((epoch_idx + 1) % val_every == 0 or epoch_idx == max_epochs - 1)
        val_metrics = {"loss": float("nan"), "tok_acc": float("nan")}
        if should_validate:
            val_metrics = evaluate_encoder_loss(local_model, local_loss_fn, val_dataloader)
            val_loss = val_metrics["loss"]
            if val_loss < best_val - min_delta:
                best_val = val_loss
                best_epoch = epoch_idx + 1
                stale_checks = 0
                save_encoder_checkpoint(ckpt_path, local_model, best_epoch, val_metrics, SPLIT_TAG)
            else:
                stale_checks += 1
            if patience is not None and stale_checks >= patience:
                progress.set_description(f"{run_name} early_stop epoch={epoch_idx+1} best_val={best_val:.6f}")
                break

        row = {
            "run": run_name,
            "epoch": epoch_idx + 1,
            "train_loss": train_metrics["loss"],
            "train_tok_acc": train_metrics["tok_acc"],
            "val_loss": val_metrics["loss"],
            "val_tok_acc": val_metrics["tok_acc"],
            "best_val_loss": best_val,
            "best_epoch": best_epoch,
            "split_tag": SPLIT_TAG,
        }
        history.append(row)
        progress.set_description(
            f"{run_name} train={train_metrics['loss']:.4f} val={val_metrics['loss']:.4f} best={best_val:.4f}"
        )

    if ckpt_path.exists():
        load_encoder_checkpoint(ckpt_path, local_model)
    else:
        save_encoder_checkpoint(ckpt_path, local_model, max_epochs, {"loss": float("nan")}, SPLIT_TAG)
        best_epoch = max_epochs

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_{run_name}_cv_es_history.csv", index=False)
    result = {
        "run": run_name,
        "best_val_loss": float(best_val),
        "best_epoch": int(best_epoch),
        "epochs_ran": int(hist_df["epoch"].max()) if not hist_df.empty else 0,
        "checkpoint": str(ckpt_path),
        "split_tag": SPLIT_TAG,
    }
    return local_model, local_loss_fn, hist_df, result


cv_results = []
cv_histories = {}
if RUN_FULL_CV:
    for cv_val_fold in CV_VAL_FOLDS:
        ctx = build_split_context(
            test_fold=TEST_FOLD,
            val_fold=cv_val_fold,
            tag_suffix=f"cv_val{cv_val_fold}",
        )
        model_cv, loss_fn_cv, hist_df, result = train_encoder_split(
            run_name=f"cv_val{cv_val_fold}",
            ctx=ctx,
        )
        result["val_fold"] = int(cv_val_fold)
        cv_results.append(result)
        cv_histories[cv_val_fold] = hist_df

cv_results_df = pd.DataFrame(cv_results)
if not cv_results_df.empty:
    cv_results_df.to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_cv_es_results.csv", index=False)
    display(cv_results_df)

if RUN_FINAL_TRAIN_ON_TRAINVAL:
    if not cv_results_df.empty and cv_results_df["best_epoch"].gt(0).any():
        FINAL_EPOCHS = int(np.median(cv_results_df.loc[cv_results_df["best_epoch"] > 0, "best_epoch"]))
    else:
        FINAL_EPOCHS = MAX_EPOCHS
    final_ctx = build_split_context(
        test_fold=TEST_FOLD,
        val_fold=None,
        tag_suffix="final_trainval",
    )
    model, loss_fn, final_history, final_result = train_encoder_split(
        run_name="final_trainval",
        ctx=final_ctx,
        max_epochs=FINAL_EPOCHS,
        patience=None,
        val_every=FINAL_EPOCHS,
    )
    optim = None
    final_history.to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_cv_es_final_trainval_history.csv", index=False)
    pd.DataFrame([final_result]).to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_cv_es_final_trainval_result.csv", index=False)
else:
    default_ctx = build_split_context(test_fold=TEST_FOLD, val_fold=VAL_FOLD, tag_suffix="single")
    model, loss_fn, loss_history_df, single_result = train_encoder_split(run_name="single", ctx=default_ctx)

loss_arr = final_history["train_loss"].tolist() if RUN_FINAL_TRAIN_ON_TRAINVAL else loss_history_df["train_loss"].tolist()
print("Current split after training:")
print(split_summary.to_string(index=False))
print("leak_counts:", split_info["leak_counts"])


In [ ]:
model.eval()
results = []
origin = []

EVAL_SPLIT_NAME = "test"
EVAL_DATASET = test_dataset
EVAL_DATALOADER = test_dataloader

print(EVAL_SPLIT_NAME, len(EVAL_DATASET))
with torch.no_grad():
    for (smiles_enc, smiles_dec_input, smiles_dec_output, cond_scalar) in EVAL_DATALOADER:
        smiles_enc = smiles_enc.to(device)
        smiles_dec_input = smiles_dec_input.to(device)
        smiles_dec_output = smiles_dec_output.to(device)
        cond_scalar = cond_scalar.to(device)

        attn_mask = smiles_dec_input == PAD_IDX
        result = model(smiles_dec_input, cond_scalar, attn_mask=attn_mask)

        results.append(result)
        origin.append(smiles_dec_output)

results = torch.cat(results, dim=0)
origin = torch.cat(origin, dim=0)
results = nn.functional.softmax(results, dim=-1)
argmax_indices = torch.argmax(results, dim=-1)
output = torch.nn.functional.one_hot(argmax_indices, num_classes=results.size(-1))

from sklearn.metrics import mean_absolute_error

results_smiles = []
origin_smiles = []

for row in argmax_indices:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    results_smiles.append(smiles or "")

for row in origin:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    origin_smiles.append(smiles or "")


In [ ]:
origin_smiles = [smiles.removesuffix("EOS").strip() for smiles in origin_smiles]
results_smiles = [smiles.removesuffix("EOS").strip() for smiles in results_smiles]

for i in range(len(results_smiles)):
    if(origin_smiles[i] != results_smiles[i]):
        print(i, "번째 다름!")
    print("real smiles      : ", origin_smiles[i])
    print("predicted smiles : ", results_smiles[i])


MAE = mean_absolute_error(origin.cpu(), torch.argmax(results.cpu(), dim=-1))
print("MAE : ", MAE)



In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import DataStructs, rdFingerprintGenerator
RDLogger.DisableLog('rdApp.error')

generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def tanimoto_similarity(smiles1, smiles2):
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    fp1 = generator.GetFingerprint(mol1)
    fp2 = generator.GetFingerprint(mol2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

def is_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None

TS = 0.0
canbe = 0
notbe = 0

for sm, orig in zip(results_smiles, origin_smiles):
    if(len(sm)==0):
        notbe += 1
        continue
    try:
        sm = PS(sm).canonicalize.psmiles
        if is_valid(sm) and is_valid(orig):
            sim = tanimoto_similarity(sm, orig)
            TS += sim
            canbe += 1
        else:
            notbe += 1
    except:
        notbe += 1
        continue


if canbe > 0:
    print("Tanimoto Similarity : ", TS / canbe)
else:
    print("No valid molecules to compare.")

print("가능한 분자 개수 :", canbe)
print("불가능한 분자 개수 :", notbe)
print("Valid fraction      :", canbe / len(results_smiles))


In [ ]:
variant_name = os.environ.get("MODELS_VARIANT", mode)
out_dir = PROJECT_ROOT / "checkpoints" / "cv_es" / variant_name
out_dir.mkdir(parents=True, exist_ok=True)
save_path = out_dir / f"encoder_only_{variant_name}_cv_es.pth"
torch.save(model.state_dict(), save_path)
print("saved:", save_path)
